# 04 - Fine-tuning transformers (E5 - E8)

Run this on Colab or Kaggle with a GPU (Runtime -> Change runtime type -> T4 GPU).

Main question: Nigerian Pidgin takes most of its words from English. So is an English model good
enough, or does pretraining on African languages help?

| ID | Model | Pretrained on |
|----|-------|---------------|
| E5 | `roberta-base` | English only |
| E6 | `xlm-roberta-base` | 100 languages (Pidgin not one of them) |
| E7 | `afro-xlmr-base` | XLM-R further trained on 17 African languages + 3 high resource ones, Pidgin included |
| E8 | `afriberta_large` | trained from scratch on 11 African languages, Pidgin included |

E6 vs E7 is the cleanest comparison: same architecture and size, the only difference is the extra
African language training. Every model gets the same data, settings and 3 seeds.

## Setup

In [ ]:
# on colab: clone my repo and install what's missing (transformers and torch are already there)
import os
REPO_URL = "https://github.com/Divineitu/pidgin-sentiment.git"
if not os.path.exists("../src") and not os.path.exists("pidgin-sentiment"):
    !git clone {REPO_URL}
if os.path.exists("pidgin-sentiment"):
    %cd pidgin-sentiment/notebooks
!pip install -q accelerate

In [ ]:
import sys
import random
import shutil

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)
from sklearn.utils.class_weight import compute_class_weight

sys.path.append("../src")
from prepare_data import prepare
from preprocess import LABELS, LABEL2ID
from metrics import get_scores, print_report, plot_confusion, log_result

if not os.path.exists("../data/processed/train.csv"):
    prepare()

train = pd.read_csv("../data/processed/train.csv")
dev = pd.read_csv("../data/processed/dev.csv")
test_official = pd.read_csv("../data/processed/test_official.csv")
test_clean = pd.read_csv("../data/processed/test_clean.csv")
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, this will be slow")

## Settings

These are the usual fine-tuning values from the BERT paper range (lr 2e-5 to 5e-5, a few epochs).
Tweets are short, so max length 64 tokens is enough and keeps training fast.

In [ ]:
MODELS = {
    "E5": ("roberta-base (english)", "FacebookAI/roberta-base"),
    "E6": ("xlm-roberta-base (multilingual)", "FacebookAI/xlm-roberta-base"),
    "E7": ("afro-xlmr-base (african)", "Davlan/afro-xlmr-base"),
    "E8": ("afriberta-large (african)", "castorini/afriberta_large"),
}
SEEDS = [42, 1, 7]
MAX_LEN = 64
LR = 2e-5
EPOCHS = 5
BATCH = 32

weights = compute_class_weight("balanced", classes=np.array(LABELS), y=train["label"])
class_weights = torch.tensor(weights, dtype=torch.float)
print(dict(zip(LABELS, weights.round(2))))

## Training

The Trainer normally uses plain cross entropy. I override `compute_loss` so the loss uses the class
weights, same as the baselines and the BiLSTM. After every epoch it checks dev macro-F1 and at the end
it keeps the best epoch.

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def to_dataset(df, tokenizer):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "labels": [LABEL2ID[l] for l in df["label"]]})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return get_scores([LABELS[i] for i in labels], [LABELS[i] for i in preds])


def predict(trainer, ds):
    logits = trainer.predict(ds).predictions
    return [LABELS[i] for i in logits.argmax(axis=-1)]


def fine_tune(model_id, seed, out_dir):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.model_max_length = 512  # afriberta's tokenizer doesn't set this
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)

    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=64,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(),
        seed=seed,
        report_to="none",
        logging_strategy="epoch",
    )
    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=to_dataset(train, tokenizer), eval_dataset=to_dataset(dev, tokenizer),
        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer, tokenizer

## Run all four models x 3 seeds

In [ ]:
all_scores = []
best_run = {"dev_f1": -1}

for exp_id, (name, model_id) in MODELS.items():
    for seed in SEEDS:
        print(f"===== {exp_id} {name} seed {seed} =====")
        trainer, tokenizer = fine_tune(model_id, seed, f"../models/tmp_{exp_id}_{seed}")
        splits = {"dev": dev, "test_official": test_official, "test_clean": test_clean}
        for split, df in splits.items():
            scores = get_scores(df["label"], predict(trainer, to_dataset(df, tokenizer)))
            all_scores.append({"exp_id": exp_id, "model": name, "seed": seed, "split": split, **scores})
            if split == "dev" and scores["macro_f1"] > best_run["dev_f1"]:
                # keep the single best run (picked on dev) for the app
                trainer.save_model("../models/best")
                tokenizer.save_pretrained("../models/best")
                best_run = {"dev_f1": scores["macro_f1"], "exp_id": exp_id, "seed": seed}
        del trainer
        shutil.rmtree(f"../models/tmp_{exp_id}_{seed}", ignore_errors=True)  # checkpoints are big, free the disk
        torch.cuda.empty_cache()

all_scores = pd.DataFrame(all_scores)
all_scores.to_csv("../results/transformer_runs.csv", index=False)
print("best run on dev:", best_run)

## Results

In [ ]:
summary = all_scores.groupby(["exp_id", "model", "split"])[["accuracy", "macro_f1", "weighted_f1",
          "f1_negative", "f1_neutral", "f1_positive"]].agg(["mean", "std"]).round(4)
macro = all_scores.groupby(["exp_id", "model", "split"])["macro_f1"].agg(["mean", "std"]).round(4).unstack("split")
macro

In [ ]:
# write the means into the shared experiment log
means = all_scores.drop(columns="seed").groupby(["exp_id", "model", "split"]).mean().round(4).reset_index()
stds = all_scores.groupby(["exp_id", "split"])["macro_f1"].std().round(4)
for _, row in means.iterrows():
    scores = row.drop(["exp_id", "model", "split"]).to_dict()
    log_result(row["exp_id"], row["model"], row["split"], scores,
               f"mean of seeds {SEEDS}, macro_f1 std={stds[(row['exp_id'], row['split'])]}")
pd.read_csv("../results/experiments.csv").query("split == 'test_clean'")[["exp_id", "model", "macro_f1", "weighted_f1", "f1_neutral"]]

## Best model: report and confusion matrix on the clean test

In [ ]:
best_tok = AutoTokenizer.from_pretrained("../models/best")
best_model = AutoModelForSequenceClassification.from_pretrained("../models/best")
best_trainer = Trainer(model=best_model, data_collator=DataCollatorWithPadding(best_tok))
pred = predict(best_trainer, to_dataset(test_clean, best_tok))
print_report(test_clean["label"], pred)
plot_confusion(test_clean["label"], pred, f"best ({best_run['exp_id']}) on clean test", "confusion_best_test_clean.png")

# save predictions for the error analysis notebook
out = test_clean.copy()
out["pred"] = pred
out.to_csv("../results/best_predictions_test_clean.csv", index=False)

## Push the best model to the Hugging Face Hub (for the app)

Needs a write token from huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HUB_ID = "Divineitu/pidgin-sentiment"  # change to your HF username
best_model.push_to_hub(HUB_ID)
best_tok.push_to_hub(HUB_ID)

In [ ]:
# download the results so they can go in the repo
!zip -r results.zip ../results
from google.colab import files
files.download("results.zip")